# Function Transformer
Log, reciprocal, square and square-root transforms on the Titanic `Age` and `Fare` columns, checked with skewness, Q-Q plots and model accuracy.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

# one look for every chart: Latin Modern font, white background
FONT = dict(family="Latin Modern Roman", size=15)

## Load the data
Three columns of the Titanic training file: the target `Survived` and two numerical inputs, `Age` and `Fare`.

In [ ]:
df = pd.read_csv("data/titanic_train.csv", usecols=["Age", "Fare", "Survived"])
df.head()

In [ ]:
# Age has 177 missing values; transforms and models need none
df.isnull().sum()

In [ ]:
# fill each missing age with the mean age (assign back: inplace=True on one column is deprecated)
df["Age"] = df["Age"].fillna(df["Age"].mean())
df.isnull().sum().sum()

## Split first

In [ ]:
X = df[["Age", "Fare"]]   # inputs
y = df["Survived"]         # target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## Is each column normal? Density plot, skewness and Q-Q plot
`stats.probplot` returns the Q-Q points and the best straight line; we draw them with Plotly.

In [ ]:
def qq_points(x):
    # osm: normal quantiles, osr: the sorted data, plus slope and intercept of the best line
    (osm, osr), (slope, intercept, r) = stats.probplot(x, dist="norm")
    return osm, osr, slope * osm + intercept


def look(x, name):
    # left: histogram with density scale; right: Q-Q plot with the red reference line
    osm, osr, line = qq_points(x)
    fig = make_subplots(1, 2, subplot_titles=[f"{name}: skewness {pd.Series(x).skew():.2f}", f"{name}: Q-Q plot"])
    fig.add_trace(go.Histogram(x=x, histnorm="probability density", nbinsx=40, marker_color="#4C78A8"), 1, 1)
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", marker=dict(color="#4C78A8", size=4)), 1, 2)
    fig.add_trace(go.Scatter(x=osm, y=line, mode="lines", line=dict(color="#E45756")), 1, 2)
    fig.update_xaxes(title_text="normal quantiles", row=1, col=2)
    fig.update_layout(template="simple_white", showlegend=False, font=FONT, height=380)
    fig.show()

In [ ]:
# Age: close to normal (skewness 0.36), points mostly on the line
look(X_train["Age"].values, "Age")

In [ ]:
# Fare: strongly right-skewed (skewness 4.88), points curve away from the line
look(X_train["Fare"].values, "Fare")

## Baseline: no transform

In [ ]:
# random_state fixes the decision tree's tie-breaking, so the numbers repeat
lr = LogisticRegression()
dt = DecisionTreeClassifier(random_state=0)
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
print("Accuracy LR", accuracy_score(y_test, lr.predict(X_test)))
print("Accuracy DT", accuracy_score(y_test, dt.predict(X_test)))

## Log transform on both columns

In [ ]:
# func: the formula to apply to every value; log1p(x) = log(1 + x) works even when x = 0
trf = FunctionTransformer(func=np.log1p)
X_train_transformed = trf.fit_transform(X_train)
X_test_transformed = trf.transform(X_test)
X_train_transformed.head()

In [ ]:
lr = LogisticRegression()
dt = DecisionTreeClassifier(random_state=0)
lr.fit(X_train_transformed, y_train)
dt.fit(X_train_transformed, y_train)
print("Accuracy LR", accuracy_score(y_test, lr.predict(X_test_transformed)))
print("Accuracy DT", accuracy_score(y_test, dt.predict(X_test_transformed)))

## Cross-validation: a more reliable score
10 different train/test splits, then the average accuracy.

In [ ]:
# put the transformer inside a pipeline, so each fold refits it on its own training part (no leakage)
from sklearn.pipeline import make_pipeline
print("raw LR", cross_val_score(LogisticRegression(), X, y, scoring="accuracy", cv=10).mean())
print("log LR", cross_val_score(make_pipeline(FunctionTransformer(np.log1p), LogisticRegression()), X, y, scoring="accuracy", cv=10).mean())
print("log DT", cross_val_score(make_pipeline(FunctionTransformer(np.log1p), DecisionTreeClassifier(random_state=0)), X, y, scoring="accuracy", cv=10).mean())

## Before and after the log, column by column

In [ ]:
# Fare: skewness drops from 4.88 to 0.40; the points hug the line
look(X_train_transformed["Fare"].values, "log(1 + Fare)")

In [ ]:
# Age: skewness goes from 0.36 to -2.24; the log made it worse
look(X_train_transformed["Age"].values, "log(1 + Age)")

## Log transform on Fare only, with ColumnTransformer

In [ ]:
# log only Fare; remainder="passthrough" keeps Age unchanged (it comes out last)
trf2 = ColumnTransformer([("log", FunctionTransformer(np.log1p), ["Fare"])], remainder="passthrough")
X_train_transformed2 = trf2.fit_transform(X_train)
X_test_transformed2 = trf2.transform(X_test)

lr = LogisticRegression()
dt = DecisionTreeClassifier(random_state=0)
lr.fit(X_train_transformed2, y_train)
dt.fit(X_train_transformed2, y_train)
print("Accuracy LR", accuracy_score(y_test, lr.predict(X_test_transformed2)))
print("Accuracy DT", accuracy_score(y_test, dt.predict(X_test_transformed2)))

In [ ]:
# cross-validate the column transformer and the model together, as a pipeline
print("LR", cross_val_score(make_pipeline(trf2, LogisticRegression()), X, y, scoring="accuracy", cv=10).mean())
print("DT", cross_val_score(make_pipeline(trf2, DecisionTreeClassifier(random_state=0)), X, y, scoring="accuracy", cv=10).mean())

## Trying other transforms on Fare
One function: pass any formula, get the cross-validated accuracy and the Fare Q-Q plot before and after.

In [ ]:
def apply_transform(transform, name):
    trf = ColumnTransformer([("t", FunctionTransformer(transform), ["Fare"])], remainder="passthrough")
    # max_iter=1000: the square makes huge numbers, and the default 100 steps are not enough
    pipe = make_pipeline(trf, LogisticRegression(max_iter=1000))
    acc = cross_val_score(pipe, X, y, scoring="accuracy", cv=10).mean()
    # transformed Fare for the skewness and the Q-Q plot only
    X_trans = trf.fit_transform(X)
    print(f"{name}: accuracy {acc:.4f}, skewness {pd.Series(X_trans[:, 0]).skew():.2f}")
    look(X_trans[:, 0], name)

In [ ]:
# no transform: the identity function gives the baseline
apply_transform(lambda x: x, "no transform")

In [ ]:
apply_transform(np.log1p, "log(1 + x)")

In [ ]:
# 1 / x fails: 15 fares are 0, and 1 / 0 is infinity. Adding 0.1 avoids it.
apply_transform(lambda x: 1 / (x + 0.1), "reciprocal 1 / (x + 0.1)")

In [ ]:
# square: meant for left-skewed data; on right-skewed Fare it makes things worse
apply_transform(np.square, "square x^2")

In [ ]:
apply_transform(np.sqrt, "square root")

In [ ]:
# any function works, even one with no sense behind it
apply_transform(np.sin, "sin(x)")